In [1]:
from vllm import LLM, SamplingParams

/home/mmitrovich/sber_new/Proj1_few-shot/llm4tab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
llm = LLM(model="facebook/opt-125m", gpu_memory_utilization=0.3)

INFO 02-03 09:16:03 [utils.py:263] non-default args: {'gpu_memory_utilization': 0.3, 'disable_log_stats': True, 'model': 'facebook/opt-125m'}


INFO 02-03 09:16:04 [model.py:530] Resolved architecture: OPTForCausalLM
INFO 02-03 09:16:04 [model.py:1545] Using max model len 2048


2026-02-03 09:16:04,655	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 02-03 09:16:04 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-03 09:16:04 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 02-03 09:16:04 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
(EngineCore_DP0 pid=3276455) INFO 02-03 09:16:05 [core.py:97] Initializing a V1 LLM engine (v0.14.1) with config: model='facebook/opt-125m', speculative_config=None, tokenizer='facebook/opt-125m', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, d

Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  5.99it/s]
Loading pt checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  5.91it/s]
(EngineCore_DP0 pid=3276455) 


(EngineCore_DP0 pid=3276455) INFO 02-03 09:16:08 [default_loader.py:291] Loading weights took 0.18 seconds
(EngineCore_DP0 pid=3276455) INFO 02-03 09:16:09 [gpu_model_runner.py:3905] Model loading took 0.24 GiB memory and 0.891636 seconds
(EngineCore_DP0 pid=3276455) INFO 02-03 09:16:11 [backends.py:644] Using cache directory: /home/mmitrovich/.cache/vllm/torch_compile_cache/10db35b806/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3276455) INFO 02-03 09:16:11 [backends.py:704] Dynamo bytecode transform time: 1.39 s
(EngineCore_DP0 pid=3276455) INFO 02-03 09:16:11 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 0.287 s
(EngineCore_DP0 pid=3276455) INFO 02-03 09:16:11 [monitor.py:34] torch.compile takes 1.68 s in total
(EngineCore_DP0 pid=3276455) INFO 02-03 09:16:12 [gpu_worker.py:358] Available KV cache memory: 23.02 GiB
(EngineCore_DP0 pid=3276455) INFO 02-03 09:16:12 [kv_cache_utils.py:1305] GPU KV cache size: 67

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:00<00:00, 71.35it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 86.54it/s]


In [1]:
import pandas as pd
import json

In [8]:
from pydantic import BaseModel, field_validator
from typing import List, Dict
from pathlib import Path


class ExperimentConfig(BaseModel):
    random_states: List[int]
    serialization_list: List[str]


class ParamsConfig(BaseModel):
    SHOTS_LIST: List[int]
    TOKEN_DICT: Dict[str, int]


class DataConfig(BaseModel):
    DATASET_NAME: str
    DATASET_PATH_IF_SYNT: str

    @field_validator("DATASET_PATH_IF_SYNT", mode="after")
    def append_dataset_name(cls, v, info):
        name = info.data["DATASET_NAME"]
        return str(Path(v) / f"{name}.csv")


class Config(BaseModel):
    experiment: ExperimentConfig
    params: ParamsConfig
    data: DataConfig


In [9]:
import yaml

with open("config.yaml") as f:
    raw_config = yaml.safe_load(f)

config = Config(**raw_config)

ValidationError: 4 validation errors for Config
experiment.serialization_list
  Field required [type=missing, input_value={'random_states': [864, 4...703, 24, 401, 210, 736]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
params
  Field required [type=missing, input_value={'model': {'name': 'deeps...03, 24, 401, 210, 736]}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
data.DATASET_NAME
  Field required [type=missing, input_value={'train_path': 'datasets/...label_column': 'target'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
data.DATASET_PATH_IF_SYNT
  Field required [type=missing, input_value={'train_path': 'datasets/...label_column': 'target'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [29]:
from pydantic import BaseModel, ConfigDict, Field
from typing import List


class ModelConfig(BaseModel):
    name: str
    max_seq_length: int = Field(gt=0)
    gpu_memory_utilization: float = Field(ge=0.0, le=1.0)


class DataConfig(BaseModel):
    train_path: str
    test_path: str
    label_column: str


class LoggingConfig(BaseModel):
    level: str
    file: str


class GPUConfig(BaseModel):
    device: str


class ExperimentConfig(BaseModel):
    random_states: List[int]


class Config(BaseModel):
    #allow controlled mutation if needed later
    model_config = ConfigDict(validate_assignment=True)

    model: ModelConfig
    data: DataConfig
    logging: LoggingConfig
    gpu: GPUConfig
    experiment: ExperimentConfig


In [23]:
import yaml

with open("config.yaml") as f:
    raw = yaml.safe_load(f)

config = Config(**raw)


In [24]:
config

Config(model=ModelConfig(name='deepseek-ai/deepseek-llm-7b-chat', max_seq_length=600, gpu_memory_utilization=0.5), data=DataConfig(train_path='datasets/train.csv', test_path='datasets/test.csv', label_column='target'), logging=LoggingConfig(level='INFO', file='app.log'), gpu=GPUConfig(device='0'), experiment=ExperimentConfig(random_states=[864, 460, 142, 629, 761, 703, 24, 401, 210, 736]))

In [25]:
config.model.name
config.model.max_seq_length

config.data.train_path
config.logging.level

for seed in config.experiment.random_states:
    print(seed)

864
460
142
629
761
703
24
401
210
736


In [26]:
config.model.name = 'd'

In [27]:
config.model.name

'd'

In [28]:
config

Config(model=ModelConfig(name='d', max_seq_length=600, gpu_memory_utilization=0.5), data=DataConfig(train_path='datasets/train.csv', test_path='datasets/test.csv', label_column='target'), logging=LoggingConfig(level='INFO', file='app.log'), gpu=GPUConfig(device='0'), experiment=ExperimentConfig(random_states=[864, 460, 142, 629, 761, 703, 24, 401, 210, 736]))

In [19]:
config.experiment.random_states

[864, 460, 142, 629, 761, 703, 24, 401, 210, 736]

In [12]:
import openml


name = 'cancer'
openml_dfs = {
'telco': 42178,
'bioresponse': 4134,
'pc4': 1049,
'spambase': 44,
'tae': 955,
'dmft': 1014,
'irish': 451,
'compas': 42193,
'crime': 43891,
'vote': 56,
'fraud': 1597,
'cancer': 15,
'steel': 1504,
'creditcard': 29,
'kc1': 1067}

dataset = openml.datasets.get_dataset(openml_dfs[name])
X, y, _, _ = dataset.get_data(target=dataset.default_target_attribute)
rskf = RepeatedStratifiedKFold(n_splits=3, n_repeats=10, random_state=42)

NameError: name 'RepeatedStratifiedKFold' is not defined

In [2]:
import os

folder_path = "/home/mmitrovich/sber_new/Proj1_few-shot/llm4tab/datasets"  # e.g., "./data"
csv_files = [f.lower().split('.')[0] for f in os.listdir(folder_path)]

print(csv_files)

['loka', 'airbnb']


In [ ]:
.lower().split('/')[0]

In [1]:
from vllm import LLM, SamplingParams
import os
os.environ["VLLM_ALLOW_LONG_MAX_MODEL_LEN"] = "1"


# hf_overrides = {
#     "rope_parameters": {
#         "rope_theta": 1000000, 
#         "rope_type": "yarn",
#         "factor": 4,
#         "original_max_position_embeddings": 8196,
#     },
#     "max_model_len": 32000, # 32768 * 4


hf_overrides = {
    "rope_parameters": {
        "rope_theta": 10000, 
        "rope_type": "yarn",
        "factor": 7.812,
        "original_max_position_embeddings": 4096, # 8196,
    },
    "max_model_len": 32000, # 32768 * 4
}

llm = LLM(model="deepseek-ai/deepseek-llm-7b-chat", gpu_memory_utilization=0.45, hf_overrides=None)

#google/gemma-7b-it
#deepseek-ai/deepseek-llm-7b-chat

/home/mmitrovich/sber_new/Proj1_few-shot/llm4tab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 02-03 15:37:19 [utils.py:263] non-default args: {'gpu_memory_utilization': 0.45, 'disable_log_stats': True, 'model': 'deepseek-ai/deepseek-llm-7b-chat'}
INFO 02-03 15:37:20 [model.py:530] Resolved architecture: LlamaForCausalLM
INFO 02-03 15:37:20 [model.py:1545] Using max model len 4096


2026-02-03 15:37:21,067	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 02-03 15:37:21 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-03 15:37:21 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 02-03 15:37:21 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:22 [core.py:97] Initializing a V1 LLM engine (v0.14.1) with config: model='deepseek-ai/deepseek-llm-7b-chat', speculative_config=None, tokenizer='deepseek-ai/deepseek-llm-7b-chat', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, 

Loading pt checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading pt checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.57s/it]
Loading pt checkpoint shards: 100% Completed | 2/2 [00:09<00:00,  5.00s/it]
Loading pt checkpoint shards: 100% Completed | 2/2 [00:09<00:00,  4.64s/it]
(EngineCore_DP0 pid=3525484) 


(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:35 [default_loader.py:291] Loading weights took 9.29 seconds
(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:35 [gpu_model_runner.py:3905] Model loading took 12.87 GiB memory and 10.093116 seconds
(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:40 [backends.py:644] Using cache directory: /home/mmitrovich/.cache/vllm/torch_compile_cache/7000f36cfa/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:40 [backends.py:704] Dynamo bytecode transform time: 4.73 s
(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:42 [backends.py:261] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:44 [backends.py:278] Compiling a graph for compile range (1, 8192) takes 2.04 s
(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:44 [monitor.py:34] torch.compile takes 6.77 s in total
(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:45 [gpu_worker.py:358] Available KV cache memory: 21.75 GiB
(En

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 27.20it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 35.25it/s]


(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:49 [gpu_model_runner.py:4856] Graph capturing finished in 4 secs, took 0.49 GiB
(EngineCore_DP0 pid=3525484) INFO 02-03 15:37:49 [core.py:273] init engine (profile, create kv cache, warmup model) took 14.08 seconds
INFO 02-03 15:37:50 [llm.py:347] Supported tasks: ['generate']


In [4]:
# prompt = """
#  <｜begin▁of▁sentence｜> You are the expert in statistics. Your task is to make classification prediction of tabular data for the given features. Your answer MUST be just 0 or 1!

# User:  You are the expert in statistics. Your task is to make classification prediction of tabular data for the given features. Your answer MUST be just 0 or 1!
#  For the given input features of airbnb listings (houses and flats available for rent) you need to predict whether it has good rating (1) or bad rating (0). Your output should be just a single number representing a binary class prediction: 0 (the rating of rental property is bad) or 1 (the rating of rental property is good). Do not predict any other tokens, only 0 or 1.

# **PREDICT** Features are: listing_id = 36299, listing_name = Kew Gardens 3BR house in cul-de-sac, listing_type = Entire townhouse, room_type = entire_home, photos_count = 20, host_id = 155938, host_name = Geert, superhost = False, latitude = 51, longitude = 0, guests = 5, bedrooms = 3, beds = 3, baths = 1, registration = False, amenities = High chair,Children’s books and toys,Wifi,Kitchen,Pack ’n play/Travel crib,Hot water,Bed linens,Extra pillows and blankets,Ethernet connection,Microwave,Coffee maker,Refrigerator,Dishwasher,Dishes and silverware,Heating,Cooking basics,Oven,Stove,Washer,Dryer,Smoke alarm,Carbon monoxide alarm,Backyard,Fire extinguisher,Essentials,Shampoo,Hangers,Hair dryer,Iron,Dedicated workspace,Private entrance,Bathtub, instant_book = False, professional_management = False, min_nights = 7, cancellation_policy = Strict, currency = GBP, cleaning_fee = 0, extra_guest_fee = 0, num_reviews = 116, rating_overall = 4, rating_accuracy = 4, rating_checkin = 4, rating_cleanliness = 4, rating_communication = 5, rating_location = 4, rating_value = 4, ttm_avg_rate = 333, l90d_avg_rate = 337. Answer is  .

# Assistant:"""


prompt = """
 <｜begin▁of▁sentence｜> You are the expert in statistics. Your task is to make classification prediction of tabular data for the given features. Return only JSON. Schema: {'class': '0 or 1'}

User:  You are the expert in statistics. Your task is to make classification prediction of tabular data for the given features. Return only JSON. Schema: {'class': '0 or 1'}
 For the given input features of airbnb listings (houses and flats available for rent) you need to predict whether it has good rating (1) or bad rating (0). Your output should be just a single number representing a binary class prediction: 0 (the rating of rental property is bad) or 1 (the rating of rental property is good). Do not predict any other tokens, only 0 or 1.

**EXAMPLES** Features are: listing_id = 384780, listing_name = Stunning Bright flat, listing_type = Entire rental unit, room_type = entire_home, photos_count = 35, host_id = 1920828, host_name = Marcus, superhost = False, latitude = 51, longitude = 0, guests = 5, bedrooms = 1, beds = 1, baths = 1, registration = False, amenities = TV,Washer,Smoke alarm,Wifi,Carbon monoxide alarm,First aid kit,Fire extinguisher,Kitchen,Essentials,Shampoo,Hair dryer,Microwave,Coffee maker,Refrigerator,Dishwasher,Dishes and silverware,Heating,Cooking basics,Oven, instant_book = False, professional_management = False, min_nights = 4, cancellation_policy = Firm, currency = GBP, cleaning_fee = 143, extra_guest_fee = 0, num_reviews = 50, rating_overall = 4, rating_accuracy = 4, rating_checkin = 4, rating_cleanliness = 4, rating_communication = 4, rating_location = 4, rating_value = 4, ttm_avg_rate = 222, l90d_avg_rate = 213. Answer is 0.
Features are: listing_id = 92399, listing_name = modern self contained flat islington, listing_type = Entire guest suite, room_type = entire_home, photos_count = 58, host_id = 497366, host_name = Helen, superhost = True, latitude = 51, longitude = 0, guests = 2, bedrooms = 1, beds = 1, baths = 1, registration = False, amenities = TV,Wifi,Hot water kettle,Paid parking off premises,Books and reading material,Conditioner,Cleaning products,Heating,Clothing storage,Wine glasses,Washer,Smoke alarm,Carbon monoxide alarm,Fire extinguisher,Essentials,Shampoo,Lock on bedroom door,Hangers,Hair dryer,Iron,Dedicated workspace,Freezer,Private entrance,Children’s books and toys,Blender,Crib,Pack ’n play/Travel crib,Room-darkening shades,Hot water,Body soap,Bed linens,Extra pillows and blankets,Microwave,Coffee maker,Refrigerator,Dishes and silverware,Cooking basics,Oven,EV charger,Shower gel,Long term stays allowed,Cleaning before checkout,Toaster, instant_book = False, professional_management = False, min_nights = 2, cancellation_policy = Moderate, currency = GBP, cleaning_fee = 40, extra_guest_fee = 26, num_reviews = 340, rating_overall = 4, rating_accuracy = 4, rating_checkin = 5, rating_cleanliness = 4, rating_communication = 5, rating_location = 4, rating_value = 4, ttm_avg_rate = 127, l90d_avg_rate = 148. Answer is 1.
Features are: listing_id = 519660, listing_name = 'Vie de Boheme' in Central London, listing_type = Entire rental unit, room_type = entire_home, photos_count = 13, host_id = 2557253, host_name = Astrid, superhost = False, latitude = 51, longitude = 0, guests = 2, bedrooms = 1, beds = 1, baths = 1, registration = False, amenities = TV,Cable TV,Wifi,Kitchen,Hot water,Dishes and silverware,Heating,Cooking basics,Washer,Smoke alarm,Carbon monoxide alarm,Fire extinguisher,Essentials,Shampoo,Hangers,Hair dryer,Iron,Dedicated workspace, instant_book = False, professional_management = False, min_nights = 5, cancellation_policy = Strict, currency = GBP, cleaning_fee = 0, extra_guest_fee = 20, num_reviews = 55, rating_overall = 4, rating_accuracy = 4, rating_checkin = 5, rating_cleanliness = 4, rating_communication = 5, rating_location = 4, rating_value = 4, ttm_avg_rate = 173, l90d_avg_rate = 185. Answer is 0.
Features are: listing_id = 369383, listing_name = Camden Town Duplex, with Downtown City Views, listing_type = Entire rental unit, room_type = entire_home, photos_count = 51, host_id = 493497, host_name = Clive, superhost = False, latitude = 51, longitude = 0, guests = 6, bedrooms = 2, beds = 3, baths = 1, registration = False, amenities = TV,Cable TV,Wifi,Kitchen,Hot water kettle,Paid parking off premises,Portable fans,Conditioner,Cleaning products,Drying rack for clothing,Heating,Coffee,Clothing storage,Washer,Smoke alarm,Carbon monoxide alarm,First aid kit,Fire extinguisher,Essentials,Shampoo,Hangers,Hair dryer,Iron,Dedicated workspace,Freezer,Private entrance,High chair,Crib,Pack ’n play/Travel crib,Room-darkening shades,Hot water,Body soap,Bed linens,Extra pillows and blankets,Microwave,Coffee maker,Refrigerator,Dishes and silverware,Cooking basics,Oven,Stove,Shower gel,Luggage dropoff allowed,Long term stays allowed,Dining table,Toaster, instant_book = True, professional_management = False, min_nights = 1, cancellation_policy = Firm, currency = GBP, cleaning_fee = 73, extra_guest_fee = 0, num_reviews = 193, rating_overall = 4, rating_accuracy = 4, rating_checkin = 4, rating_cleanliness = 4, rating_communication = 4, rating_location = 4, rating_value = 4, ttm_avg_rate = 409, l90d_avg_rate = 507. Answer is 1.
Features are: listing_id = 416238, listing_name = 3 Bedroom Sunny, Airy Penthouse Hampstead Village!, listing_type = Entire rental unit, room_type = entire_home, photos_count = 14, host_id = 2069875, host_name = Nan And Sam, superhost = True, latitude = 51, longitude = 0, guests = 7, bedrooms = 3, beds = 3, baths = 1, registration = False, amenities = TV,Children’s books and toys,Blender,Wifi,Crib,Kitchen,Pack ’n play/Travel crib,Hot water kettle,Paid parking off premises,Portable fans,Pets allowed,Hot water,Bed linens,Extra pillows and blankets,Microwave,Cleaning products,Coffee maker,Refrigerator,Dishes and silverware,Heating,Cooking basics,Oven,Stove,Washer,Dryer,Smoke alarm,Carbon monoxide alarm,Luggage dropoff allowed,Essentials,Long term stays allowed,Shampoo,Hangers,Hair dryer,Iron,Dedicated workspace,Baking sheet,Freezer,Toaster,Baby bath, instant_book = False, professional_management = False, min_nights = 2, cancellation_policy = Strict, currency = GBP, cleaning_fee = 160, extra_guest_fee = 0, num_reviews = 110, rating_overall = 4, rating_accuracy = 4, rating_checkin = 4, rating_cleanliness = 4, rating_communication = 4, rating_location = 4, rating_value = 4, ttm_avg_rate = 731, l90d_avg_rate = 924. Answer is 0.
Features are: listing_id = 469189, listing_name = Stylish 2-bedroom garden flat with antique charm, listing_type = Entire condo, room_type = entire_home, photos_count = 24, host_id = 1952786, host_name = Anne, superhost = True, latitude = 51, longitude = 0, guests = 3, bedrooms = 2, beds = 3, baths = 1, registration = False, amenities = TV,Wifi,Kitchen,Hot water kettle,Paid parking off premises,Books and reading material,Portable fans,Conditioner,Mini fridge,Laundromat nearby,Cleaning products,Drying rack for clothing,Heating,Clothing storage,Wine glasses,Washer,Dryer,Smoke alarm,Carbon monoxide alarm,Essentials,Shampoo,Hangers,Hair dryer,Iron,Dedicated workspace,Private entrance,Bathtub,Children’s books and toys,Hot water,Body soap,Bed linens,Extra pillows and blankets,Microwave,Coffee maker,Refrigerator,Dishwasher,Dishes and silverware,Cooking basics,Oven,Stove,Single level home,Shower gel,Patio or balcony,Backyard,Luggage dropoff allowed,Dining table,Toaster, instant_book = False, professional_management = False, min_nights = 1, cancellation_policy = Firm, currency = GBP, cleaning_fee = 13, extra_guest_fee = 0, num_reviews = 49, rating_overall = 4, rating_accuracy = 4, rating_checkin = 4, rating_cleanliness = 4, rating_communication = 5, rating_location = 4, rating_value = 4, ttm_avg_rate = 381, l90d_avg_rate = 356. Answer is 1.
Features are: listing_id = 461923, listing_name = Surprisingly Spacious & Stylish 2 Bed, 2 Bath Flat, listing_type = Entire rental unit, room_type = entire_home, photos_count = 30, host_id = 2117678, host_name = London, superhost = False, latitude = 51, longitude = 0, guests = 5, bedrooms = 2, beds = 2, baths = 2, registration = False, amenities = TV,Cable TV,Wifi,Waterfront,Kitchen,Hot water kettle,Paid parking off premises,Books and reading material,Portable fans,Conditioner,Elevator,Laundromat nearby,Cleaning products,Drying rack for clothing,Heating,Paid parking on premises,Coffee,Clothing storage,Wine glasses,Washer,Smoke alarm,First aid kit,Fire extinguisher,Essentials,Shampoo,Hangers,Iron,Dedicated workspace,Freezer,Private entrance,Bathtub,Children’s books and toys,Blender,Window guards,Table corner guards,Room-darkening shades,Hot water,Body soap,Bed linens,Extra pillows and blankets,Microwave,Coffee maker,Refrigerator,Dishwasher,Dishes and silverware,Cooking basics,Oven,Stove,EV charger,Shower gel,Luggage dropoff allowed,Long term stays allowed,Cleaning before checkout,Dining table,Outdoor playground,Baking sheet,Toaster, instant_book = False, professional_management = False, min_nights = 2, cancellation_policy = Firm, currency = GBP, cleaning_fee = 80, extra_guest_fee = 0, num_reviews = 281, rating_overall = 4, rating_accuracy = 4, rating_checkin = 4, rating_cleanliness = 4, rating_communication = 4, rating_location = 4, rating_value = 4, ttm_avg_rate = 235, l90d_avg_rate = 243. Answer is 0.
Features are: listing_id = 264789, listing_name = Huge Three Bedroom Flat with parking and terrace, listing_type = Entire rental unit, room_type = entire_home, photos_count = 42, host_id = 1389063, host_name = Sue, superhost = False, latitude = 51, longitude = 0, guests = 7, bedrooms = 3, beds = 4, baths = 2, registration = False, amenities = TV,Blender,Wifi,Pool,Crib,Kitchen,Free parking on premises,Coffee maker,Refrigerator,Drying rack for clothing,Dishwasher,Dishes and silverware,Heating,Cooking basics,Clothing storage,Stove,Washer,Dryer,Smoke alarm,Carbon monoxide alarm,Patio or balcony,First aid kit,Backyard,Fire extinguisher,Essentials,Shampoo,Dining table,Hair dryer,Iron, instant_book = True, professional_management = False, min_nights = 3, cancellation_policy = Flexible, currency = GBP, cleaning_fee = 53, extra_guest_fee = 0, num_reviews = 69, rating_overall = 4, rating_accuracy = 4, rating_checkin = 5, rating_cleanliness = 4, rating_communication = 4, rating_location = 4, rating_value = 4, ttm_avg_rate = 304, l90d_avg_rate = 323. Answer is 0.
Features are: listing_id = 300723, listing_name = Stylish 3 Bed 2 Bath Apartment Gdn + cafe culture, listing_type = Entire rental unit, room_type = entire_home, photos_count = 41, host_id = 1549137, host_name = Rita, superhost = False, latitude = 51, longitude = 0, guests = 6, bedrooms = 3, beds = 3, baths = 2, registration = False, amenities = TV,Cable TV,Wifi,Kitchen,Hot water kettle,Paid parking off premises,Books and reading material,Portable fans,Laundromat nearby,Outdoor furniture,Cleaning products,Drying rack for clothing,Heating,Coffee,Wine glasses,Washer,Smoke alarm,Fire extinguisher,Essentials,Shampoo,Hangers,Hair dryer,Iron,Dedicated workspace,Outdoor dining area,Freezer,Bathtub,Blender,Hot water,Body soap,Bed linens,Microwave,Coffee maker,Refrigerator,Fire pit,Dishwasher,Dishes and silverware,Cooking basics,Oven,Stove,BBQ grill,Patio or balcony,Backyard,Luggage dropoff allowed,Cleaning before checkout,Dining table,Baking sheet,Barbecue utensils,Toaster, instant_book = False, professional_management = False, min_nights = 3, cancellation_policy = Firm, currency = GBP, cleaning_fee = 134, extra_guest_fee = 0, num_reviews = 78, rating_overall = 4, rating_accuracy = 4, rating_checkin = 4, rating_cleanliness = 4, rating_communication = 5, rating_location = 4, rating_value = 4, ttm_avg_rate = 247, l90d_avg_rate = 267. Answer is 0.
Features are: listing_id = 52624, listing_name = Close to Wimbledon All England Tennis -huge double, listing_type = Private room in guest suite, room_type = private_room, photos_count = 9, host_id = 243610, host_name = Beverley, superhost = True, latitude = 51, longitude = 0, guests = 2, bedrooms = 1, beds = 2, baths = 1, registration = False, amenities = TV,Smoke alarm,Shower gel,Wifi,Patio or balcony,First aid kit,Backyard,Luggage dropoff allowed,Essentials,Shampoo,Paid parking off premises,Books and reading material,Hangers,Hair dryer,Hot water,Body soap,Breakfast,Conditioner,Bed linens,Extra pillows and blankets,Ethernet connection,Heating,Clothing storage, instant_book = False, professional_management = False, min_nights = 2, cancellation_policy = Strict, currency = GBP, cleaning_fee = 26, extra_guest_fee = 0, num_reviews = 20, rating_overall = 4, rating_accuracy = 5, rating_checkin = 5, rating_cleanliness = 5, rating_communication = 5, rating_location = 5, rating_value = 5, ttm_avg_rate = 125, l90d_avg_rate = 141. Answer is 0.
Features are: listing_id = 426354, listing_name = 1 bedroom flat with big balcony!, listing_type = Entire rental unit, room_type = entire_home, photos_count = 29, host_id = 2119740, host_name = André, superhost = False, latitude = 51, longitude = 0, guests = 2, bedrooms = 1, beds = 4, baths = 1, registration = False, amenities = TV,Cable TV,Wifi,Kitchen,Hot water kettle,Paid parking off premises,Portable fans,Conditioner,Elevator,Outdoor furniture,Cleaning products,Drying rack for clothing,Heating,Coffee,Clothing storage,Wine glasses,Washer,Smoke alarm,Carbon monoxide alarm,First aid kit,Essentials,Shampoo,Hangers,Hair dryer,Iron,Dedicated workspace,Freezer,Sound system,Bathtub,Blender,Hot water,Bed linens,Extra pillows and blankets,Microwave,Coffee maker,Refrigerator,Dishwasher,Dishes and silverware,Cooking basics,Oven,Shower gel,Patio or balcony,Luggage dropoff allowed,Long term stays allowed,Toaster, instant_book = False, professional_management = False, min_nights = 4, cancellation_policy = Strict, currency = GBP, cleaning_fee = 60, extra_guest_fee = 0, num_reviews = 13, rating_overall = 4, rating_accuracy = 5, rating_checkin = 5, rating_cleanliness = 5, rating_communication = 5, rating_location = 4, rating_value = 5, ttm_avg_rate = 259, l90d_avg_rate = 271. Answer is 0.
Features are: listing_id = 520350, listing_name = London Calling - Cosy two bedroom flat, King's X, listing_type = Entire condo, room_type = entire_home, photos_count = 50, host_id = 2353375, host_name = S, superhost = True, latitude = 51, longitude = 0, guests = 6, bedrooms = 2, beds = 3, baths = 1, registration = False, amenities = Wifi,Kitchen,Board games,Hot water kettle,Paid parking off premises,Books and reading material,Portable fans,Conditioner,Elevator,Laundromat nearby,Cleaning products,Drying rack for clothing,Heating,Coffee,Clothing storage,Wine glasses,Washer,Dryer,Smoke alarm,First aid kit,Essentials,Shampoo,Hangers,Hair dryer,Freezer,Sound system,Rice maker,Bathtub,Blender,Window guards,Pack ’n play/Travel crib,Hot water,Body soap,Bed linens,Extra pillows and blankets,Microwave,Coffee maker,Refrigerator,Dishes and silverware,Cooking basics,Oven,Stove,Single level home,Shower gel,Long term stays allowed,Cleaning before checkout,Dining table,Baking sheet,Toaster, instant_book = False, professional_management = False, min_nights = 2, cancellation_policy = Strict, currency = GBP, cleaning_fee = 0, extra_guest_fee = 13, num_reviews = 58, rating_overall = 4, rating_accuracy = 5, rating_checkin = 4, rating_cleanliness = 4, rating_communication = 5, rating_location = 4, rating_value = 4, ttm_avg_rate = 255, l90d_avg_rate = 379. Answer is 0.

**PREDICT** Features are: listing_id = 36299, listing_name = Kew Gardens 3BR house in cul-de-sac, listing_type = Entire townhouse, room_type = entire_home, photos_count = 20, host_id = 155938, host_name = Geert, superhost = False, latitude = 51, longitude = 0, guests = 5, bedrooms = 3, beds = 3, baths = 1, registration = False, amenities = High chair,Children’s books and toys,Wifi,Kitchen,Pack ’n play/Travel crib,Hot water,Bed linens,Extra pillows and blankets,Ethernet connection,Microwave,Coffee maker,Refrigerator,Dishwasher,Dishes and silverware,Heating,Cooking basics,Oven,Stove,Washer,Dryer,Smoke alarm,Carbon monoxide alarm,Backyard,Fire extinguisher,Essentials,Shampoo,Hangers,Hair dryer,Iron,Dedicated workspace,Private entrance,Bathtub, instant_book = False, professional_management = False, min_nights = 7, cancellation_policy = Strict, currency = GBP, cleaning_fee = 0, extra_guest_fee = 0, num_reviews = 116, rating_overall = 4, rating_accuracy = 4, rating_checkin = 4, rating_cleanliness = 4, rating_communication = 5, rating_location = 4, rating_value = 4, ttm_avg_rate = 333, l90d_avg_rate = 337. Answer is  .

Assistant:
"""



sampling_params = SamplingParams(temperature=0, logprobs=1, max_tokens=600)
outputs = llm.generate([prompt], sampling_params)
answer = outputs[0].outputs[0].text
answer

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.27s/it, est. speed input: 1628.88 toks/s, output: 74.10 toks/s]


'To make a classification prediction for the given airbnb listings, I would need to use a machine learning algorithm such as logistic regression, decision tree, or random forest. However, as an AI language model, I am unable to perform such calculations. I can only provide you with the information you have given me.\n\nBased on the features provided, here are the predictions for each listing:\n\n* listing_id = 384780, prediction = 0\n* listing_id = 52624, prediction = 1\n* listing_id = 469189, prediction = 0\n* listing_id = 426354, prediction = 1\n* listing_id = 36299, prediction = 0\n* listing_id = 461923, prediction = 0\n* listing_id = 520350, prediction = 0\n* listing_id = 26435, prediction = 0\n\nPlease consult with a data scientist or machine learning expert to implement the algorithm and make the predictions.'

In [9]:

import re
def extract_class_safe(generated_text):

    text = generated_text.lower()
    
    patterns = [
        r'"class"\s*:\s*"(\d)"',
        r"'class'\s*:\s*'(\d)'",
        r'"class"\s*:\s*(\d)',
        r'\*\*answer:\*\*\s*(\d)',      
        r'answer:\s*(\d)',             
        r'prediction:\s*(\d)',
        r'class:\s*(\d)',
        r'output:\s*(\d)',
        r'label:\s*(\d)',
        r'answer is\s*(\d)',
        r'class is\s*(\d)',
        r'output is\s*(\d)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        print(match)
        if match:
            found = match.group(1)
            if found in ['0', '1']:
                return found
    
    for char in reversed(text):
        if char in ['0', '1']:
            return char
    
    return None

In [10]:
answer = outputs[0].outputs[0].text
logprobs_list = outputs[0].outputs[0].logprobs


pred_class = extract_class_safe(answer)
pred_class = str(int(float(pred_class)))
pred_class

None
None
None
None
None
None
None
None
None
None
None
None


'0'

In [6]:
answer.lower()

'**sure, here is the classification prediction:**\n\n**input:**\nlisting_id = 36299\nlisting_name = kew gardens 3br house in cul-de-sac\nlisting_type = entire townhouse\nroom_type = entire_home\nphotos_count = 20\nhost_id = 155938\nhost_name = geert\nsuperhost = false\nlatitude = 51\nlongitude = 0\nguests = 5\nbedrooms = 3\nbeds = 3\nbaths = 1\nregistration = false\namenities = high chair,children’s books and toys,wifi,kitchen,pack ’n play/travel crib,hot water,bed linens,extra pillows and blankets,ethernet connection,microwave,coffee maker,refrigerator,dishwasher,dishes and silverware,heating,cooking basics,oven,stove,washer,dryer,smoke alarm,carbon monoxide alarm,backyard,fire extinguisher,essentials,shampoo,hangers,hair dryer,iron,dedicated workspace,private entrance,bathtub\n\n**output:**\n1\n\nthe listing has a good rating of 4, therefore the output is 1.'

In [5]:
logprobs_list

[{15: Logprob(logprob=-0.46973085403442383, rank=1, decoded_token='0')},
 {100001: Logprob(logprob=-0.0348019041121006, rank=1, decoded_token='<｜end▁of▁sentence｜>')}]

In [6]:
answer

'0'

In [6]:
logprobs_list

[{1917: Logprob(logprob=-0.14216695725917816, rank=1, decoded_token='```')},
 {7774: Logprob(logprob=-0.025714360177516937, rank=1, decoded_token='python')},
 {108: Logprob(logprob=-0.12730471789836884, rank=1, decoded_token='\n')},
 {235282: Logprob(logprob=-0.17160306870937347, rank=1, decoded_token='{')},
 {108: Logprob(logprob=-0.0077247703447937965, rank=1, decoded_token='\n')},
 {139: Logprob(logprob=-0.24236032366752625, rank=1, decoded_token='  ')},
 {235281: Logprob(logprob=-5.722029527532868e-06, rank=1, decoded_token='"')},
 {62854: Logprob(logprob=-0.006067072972655296, rank=1, decoded_token='prediction')},
 {1192: Logprob(logprob=-1.1920928244535389e-07, rank=1, decoded_token='":')},
 {235248: Logprob(logprob=-4.6132929128361866e-05, rank=1, decoded_token=' ')},
 {235274: Logprob(logprob=-8.344646857949556e-07, rank=1, decoded_token='1')},
 {108: Logprob(logprob=-5.1020273531321436e-05, rank=1, decoded_token='\n')},
 {235270: Logprob(logprob=-2.622600959512056e-06, rank=1,

In [11]:
outputs[0].outputs[0].text

'**\n\nThe answer is 1.**\n\nThe listing has a high number of positive reviews and a high rating overall.'

In [10]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import time

serialization_list = ['1ssfsfs','4dad']

df_dict = {
rs: pd.DataFrame(
    np.nan,
    index=serialization_list,
    columns=['roc_auc', 'f1', 'time']
)
for rs in [5,7,3]
}

for k, v in tqdm(df_dict.items(), total = len(df_dict)):
    for serialization in serialization_list:
        start_time = time.time()
        roc_auc = 0.4
        f1 = 131
        v.loc[serialization, 'roc_auc'] = roc_auc
        v.loc[serialization, 'f1'] = f1
        end_time = time.time()
        v.loc[serialization, 'time'] = end_time - start_time

100%|██████████| 3/3 [00:00<00:00, 1806.85it/s]


In [11]:
v

,roc_auc,f1,time
1ssfsfs,0.4,131.0,0.000198
4dad,0.4,131.0,0.000072


In [5]:
import json

json.loads("{'class': 0 or 1}")

JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)

In [7]:
import pandas as pd
from tqdm import tqdm
import numpy as np

serialization_list = ['old_1', 'old_2']
random_state_list = [10,56,3]

df_dict = {
rs: pd.DataFrame(
    np.nan,
    index=serialization_list,
    columns=['roc_auc', 'f1', 'time']
)
for rs in random_state_list
}


for k, v in tqdm(df_dict.items(), total = len(df_dict)):
    print(k)

100%|██████████| 3/3 [00:00<00:00, 27294.82it/s]

10
56
3


In [3]:
import yaml

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [5]:
config['openml']

{'credit': 363626,
 'transfusion': 363621,
 'fitness': 363671,
 'diabetes': 363629,
 'biodegr': 363696,
 'marketing': 363684}

In [2]:
from sklearn.model_selection import RepeatedStratifiedKFold
import openml
dataset = openml.datasets.get_dataset(42178)
X, y, _, _ = dataset.get_data(target=dataset.default_target_attribute)
rskf = RepeatedStratifiedKFold(n_splits=3, n_repeats=10, random_state=42)

train_idx, test_idx = next(rskf.split(X, y))

In [4]:
test_idx


array([   0,    3,    9, ..., 7034, 7038, 7042], shape=(2348,))

In [ ]:
'credit': 363626,
         'transfusion': 363621,
         'fitness': 363671,
         'diabetes': 363629,
         'biodegr': 363696,
         'marketing': 363684